# Objective

The objective of this notebook is to develop and compare multiple machine learning regression models for daily retail sales prediction using the preprocessed Rossmann dataset.

This phase focuses on building a systematic modeling workflow by establishing baseline performance, training classical machine learning algorithms, and evaluating their ability to capture complex patterns in retail sales.

The key goals of this notebook are:

- Establish a simple baseline model to measure the minimum expected performance.
- Train and evaluate multiple regression algorithms.
- Compare model performance using appropriate regression metrics.
- Understand the strengths and limitations of different modeling approaches.
- Identify the best-performing model for further evaluation and optimization.

The models developed in this phase will serve as the foundation for Phase 7, where detailed model evaluation, error analysis, feature importance analysis, and final model selection will be performed.

# Modeling Strategy

The project follows a progressive modeling approach:

1. Baseline Model
   - Establishes a reference performance level.
   - Determines whether machine learning models provide meaningful improvement.

2. Linear Regression
   - Provides an interpretable statistical baseline.
   - Helps understand the limitations of linear assumptions on retail sales data.

3. Decision Tree Regression
   - Introduces nonlinear decision boundaries.
   - Captures feature interactions and business rules.

4. Random Forest Regression
   - Uses ensemble learning to improve stability and predictive performance.
   - Handles nonlinear relationships effectively.

5. Gradient Boosting Regression
   - Sequentially improves weak learners.
   - Expected to perform strongly on structured tabular data.

Model selection will be based on validation performance while considering both predictive accuracy and interpretability.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [2]:
df=pd.read_csv("../data/processed/train_engineered.csv")
df["StateHoliday"]=df["StateHoliday"].astype(str)
df.loc[df["CompetitionDistance"].isna(),"CompetitionDistance"]= -1

df.info()

C:\Users\ragha\AppData\Local\Temp\ipykernel_21160\370352286.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("../data/processed/train_engineered.csv")


<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Store                      844392 non-null  int64  
 1   DayOfWeek                  844392 non-null  int64  
 2   Date                       844392 non-null  str    
 3   Sales                      844392 non-null  int64  
 4   Customers                  844392 non-null  int64  
 5   Open                       844392 non-null  int64  
 6   Promo                      844392 non-null  int64  
 7   StateHoliday               844392 non-null  str    
 8   SchoolHoliday              844392 non-null  int64  
 9   StoreType                  844392 non-null  str    
 10  Assortment                 844392 non-null  str    
 11  CompetitionDistance        844392 non-null  float64
 12  CompetitionOpenSinceMonth  575773 non-null  float64
 13  CompetitionOpenSinceYear   575773 non-nu

In [3]:
print(df.shape)
print(df.columns)


(844392, 34)
Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day',
       'WeekOfYear', 'Quarter', 'IsMonthStart', 'IsMonthEnd', 'IsWeekend',
       'CompetitionInfoAvailable', 'CompetitionOpenDate',
       'CompetitionAgeMonths', 'Promo2StartDate', 'PromoAgeMonths',
       'Promo2AgeMonths', 'TransactionMonth', 'IsPromoMonth'],
      dtype='str')


In [4]:
train_df = df[df["Date"] < "2015-06-01"].copy()
valid_df = df[df["Date"] >= "2015-06-01"].copy()

# 4. Feature and Target Separation

In [5]:
X_train = train_df.drop(columns="Sales")
y_train = train_df["Sales"]

X_valid = valid_df.drop(columns="Sales")
y_valid = valid_df["Sales"]

In [6]:
one_hot_features = [
    "StateHoliday",
    "StoreType",
    "Assortment"
]
numerical_features = [
    "CompetitionDistance",
    "Year",
    "CompetitionAgeMonths",
    "Promo2AgeMonths"
]
binary_features = [
    "Promo",
    "SchoolHoliday",
    "Promo2",
    "IsMonthStart",
    "IsMonthEnd",
    "IsWeekend",
    "CompetitionInfoAvailable",
    "IsPromoMonth"
]

temporal_features = [
    "DayOfWeek",
    "Month",
    "Day",
    "WeekOfYear",
    "Quarter"
]

In [7]:
columns_to_drop = [
    "Date",
    "Customers",
    "Store",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "CompetitionOpenDate",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
    "Promo2StartDate",
    "PromoAgeMonths",
    "TransactionMonth",
    "Open"
]

X_train = X_train.drop(columns=columns_to_drop)
X_valid = X_valid.drop(columns=columns_to_drop)

In [8]:
set(
    one_hot_features
    + numerical_features
    + binary_features
    + temporal_features
) == set(X_train.columns)

True

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            one_hot_features
        )
    ],
    remainder="passthrough"
)

In [10]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [11]:
print(X_train_processed.shape)
X_valid.shape

(785781, 28)


(58611, 20)

In [12]:
baseline_model=DummyRegressor(strategy="mean")
baseline_model.fit(X_train_processed,y_train)
baseline_predictions = baseline_model.predict(X_valid_processed)

In [13]:
baseline_mae=mean_absolute_error(y_valid,baseline_predictions)
baseline_rmse=root_mean_squared_error(y_valid,baseline_predictions)
baseline_r2=r2_score(y_valid,baseline_predictions)

In [14]:
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)
print("R2:", baseline_r2)

MAE: 2278.4932822630994
RMSE: 3118.937105565342
R2: -0.005014533072101424


In [15]:
results = []

results.append({
    "Model": "Dummy Regressor",
    "MAE": baseline_mae,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
})

In [16]:
from sklearn.linear_model import LinearRegression

In [17]:
linear_model=LinearRegression()

linear_model.fit(X_train_processed,y_train)
y_predict=linear_model.predict(X_valid_processed)

In [18]:
linear_mae=mean_absolute_error(y_valid,y_predict)
linear_rmse=root_mean_squared_error(y_valid,y_predict)
linear_r2=r2_score(y_valid,y_predict)

In [19]:
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R2:", linear_r2)

MAE: 1963.9179057355127
RMSE: 2660.7120354336644
R2: 0.26860016865918845


In [20]:
results.append({
    "Model": "Linear Regression",
    "MAE": linear_mae,
    "RMSE": linear_rmse,
    "R2": linear_r2
})

In [21]:
from sklearn.ensemble import RandomForestRegressor

In [22]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

In [23]:
rf_model.fit(
    X_train_processed,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [24]:
y_pred = rf_model.predict(
    X_valid_processed
)

In [25]:
rf_mae=mean_absolute_error(y_valid,y_pred)
rf_rmse=root_mean_squared_error(y_valid,y_pred)
rf_r2=r2_score(y_valid,y_pred)

In [26]:
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

MAE: 1329.7829819596827
RMSE: 1913.8064478735632
R2: 0.6215965088599233


In [27]:
results.append({
    "Model": "Random Forest",
    "MAE": rf_mae,
    "RMSE": rf_rmse,
    "R2": rf_r2
})

In [28]:
from xgboost import XGBRegressor

In [52]:

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

In [53]:
xgb_model.fit(
    X_train_processed,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [54]:
y_pred = xgb_model.predict(
    X_valid_processed
)

In [55]:
xgb_mae=mean_absolute_error(y_valid,y_pred)
xgb_rmse=root_mean_squared_error(y_valid,y_pred)
xgb_r2=r2_score(y_valid,y_pred)

In [56]:
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2:", xgb_r2)

MAE: 1136.5947265625
RMSE: 1619.7706298828125
R2: 0.7289395332336426


In [57]:
train_pred = xgb_model.predict(X_train_processed)

train_r2 = r2_score(
    y_train,
    train_pred
)

train_rmse = root_mean_squared_error(
    y_train,
    train_pred
)

print(train_r2)
print(train_rmse)

0.8383587002754211
1247.611328125


In [69]:
results.append({
    "Model": "XGBoost",
    "MAE": xgb_mae,
    "RMSE": xgb_rmse,
    "R2": xgb_r2
})


In [70]:

results_df = pd.DataFrame(results)


In [71]:
results_df.sort_values(
    by="RMSE"
)

,Model,MAE,RMSE,R2
3,XGBoost,1136.594727,1619.770630,0.728940
2,Random Forest,1329.782982,1913.806448,0.621597
1,Linear Regression,1963.917906,2660.712035,0.268600
0,Dummy Regressor,2278.493282,3118.937106,-0.005015


In [72]:
import joblib

joblib.dump(
    xgb_model,
    "../models/xgboost_model.joblib"
)

['../models/xgboost_model.joblib']

In [73]:
results_df.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

In [75]:
predictions = pd.DataFrame({
    "Actual": y_valid,
    "Predicted": y_pred
})

predictions.to_csv(
    "../reports/xgboost_predictions.csv",
    index=False
)

XGBoost achieved the best validation performance (R² = 0.729, RMSE = 1619.77) and was selected as the final forecasting model. Compared with the Dummy Regressor baseline, it reduced RMSE by approximately 48%, demonstrating the effectiveness of the engineered features and boosting-based learning for retail sales forecasting.

In [77]:
import plotly.express as px

In [78]:
fig=px.bar(results_df,x="Model",y="R2",text_auto=".2f",hover_data=["MAE","RMSE"])
fig.update_layout(xaxis_title="Models",yaxis_title="R2 Score")
fig.show()